# Notebook 03 - Visualizaciones y Storytelling

**Proyecto:** Dengue en Argentina - EDA (2018-2025)  
**Autora:** Fabiana Grisel Gonzalez  
**Input:** `data/processed/dengue_limpio.csv`  

---

## Narrativa

Este notebook construye las visualizaciones finales que cuentan la historia del dengue en Argentina:

1. **Contexto:** el dengue en Argentina a lo largo del tiempo
2. **Geografia:** donde golpea mas fuerte
3. **Estacionalidad:** cuando ocurren los brotes
4. **Poblacion:** a quienes afecta mas
5. **Conclusion:** que nos dicen los datos?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / 'src'))
from utils import configurar_graficos, guardar_figura

configurar_graficos()

# Paleta personalizada del proyecto
COLOR_DENGUE = '#E63946'
COLOR_ZIKA = '#457B9D'
COLOR_ACCENT = '#F4A261'
PALETTE_PROV = ['#E63946', '#F4A261', '#2A9D8F', '#264653', '#E9C46A']

# Variables de referencia
COL_CANTIDAD = 'cantidad_casos'
COL_SEMANA = 'semanas_epidemiologicas'
COL_ANIO = 'ano'

df = pd.read_csv('../data/processed/dengue_limpio.csv')
print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')

---
## Visualizacion 1: La historia del dengue en Argentina

Evolucion interanual con anotaciones en los brotes mas importantes.

In [ ]:
casos_anio = df.groupby(COL_ANIO)[COL_CANTIDAD].sum()

fig, ax = plt.subplots(figsize=(12, 6))

# Linea con marcadores
ax.plot(casos_anio.index, casos_anio.values, color=COLOR_DENGUE,
        linewidth=2.5, marker='o', markersize=8, zorder=5)

# Area sombreada debajo
ax.fill_between(casos_anio.index, casos_anio.values, alpha=0.15, color=COLOR_DENGUE)

# Anotar el anio pico
anio_pico = casos_anio.idxmax()
valor_pico = casos_anio.max()
ax.annotate(f'Brote record\n{valor_pico:,.0f} casos',
            xy=(anio_pico, valor_pico),
            xytext=(anio_pico - 1, valor_pico * 0.8),
            fontsize=12, fontweight='bold', color=COLOR_DENGUE,
            arrowprops=dict(arrowstyle='->', color=COLOR_DENGUE, lw=2),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLOR_DENGUE, alpha=0.9))

# Formato
ax.set_title('Evolucion del Dengue en Argentina', fontsize=18, fontweight='bold', pad=20)
ax.text(0.5, 1.02, 'Casos confirmados por anio (fuente: Ministerio de Salud)',
        transform=ax.transAxes, ha='center', fontsize=11, color='gray')
ax.set_xlabel('')
ax.set_ylabel('Casos confirmados', fontsize=12)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xticks(casos_anio.index)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
guardar_figura(fig, 'story_evolucion_dengue')
plt.show()

---
## Visualizacion 2: Mapa de calor geografico

Donde golpea mas fuerte el dengue? Heatmap de provincia x anio.

In [ ]:
# Excluir DESCONOCIDA
df_geo = df[df['provincia_nombre'] != 'DESCONOCIDA']

pivot_geo = df_geo.pivot_table(index='provincia_nombre', columns=COL_ANIO,
                                values=COL_CANTIDAD, aggfunc='sum', fill_value=0)

# Ordenar por total descendente
pivot_geo['total'] = pivot_geo.sum(axis=1)
pivot_geo = pivot_geo.sort_values('total', ascending=False).drop(columns='total')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot_geo, cmap='YlOrRd', linewidths=0.5, linecolor='white',
            annot=True, fmt='.0f', ax=ax,
            cbar_kws={'label': 'Casos confirmados', 'shrink': 0.8})
ax.set_title('Distribucion geografica del dengue por anio', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Anio', fontsize=12)
ax.set_ylabel('')
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
guardar_figura(fig, 'story_heatmap_geografico')
plt.show()

---
## Visualizacion 3: El ritmo del dengue - Estacionalidad

En que momento del anio hay que estar alerta?

In [ ]:
# Curva de estacionalidad comparando anios
estacional = df.pivot_table(index=COL_SEMANA, columns=COL_ANIO,
                            values=COL_CANTIDAD, aggfunc='sum', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))

anio_max = estacional.sum().idxmax()
for col in estacional.columns:
    alpha = 1.0 if col == anio_max else 0.4
    lw = 3 if col == anio_max else 1.2
    ax.plot(estacional.index, estacional[col], label=str(col), alpha=alpha, linewidth=lw)

# Zona de riesgo
ax.axvspan(1, 22, alpha=0.08, color='red', label='Temporada alta')

ax.set_title('Estacionalidad del dengue por semana epidemiologica',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Semana epidemiologica', fontsize=12)
ax.set_ylabel('Casos confirmados', fontsize=12)
ax.legend(title='Anio', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
guardar_figura(fig, 'story_estacionalidad')
plt.show()

---
## Visualizacion 4: A quienes afecta mas?

Distribucion por grupo de edad.

In [ ]:
edad_data = df.groupby('grupo_edad_desc')[COL_CANTIDAD].sum().sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colores = sns.color_palette('YlOrRd', len(edad_data))
ax.barh(edad_data.index, edad_data.values, color=colores)
ax.set_title('Casos de dengue por grupo de edad', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Casos confirmados', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)

# Destacar el grupo mas afectado
max_idx = len(edad_data) - 1
ax.get_children()[max_idx].set_color(COLOR_DENGUE)
ax.get_children()[max_idx].set_edgecolor('black')
ax.get_children()[max_idx].set_linewidth(1.5)

for i, v in enumerate(edad_data.values):
    ax.text(v + edad_data.max() * 0.01, i, f'{v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
guardar_figura(fig, 'story_edad')
plt.show()

---
## Visualizacion 5: Dashboard resumen

Panel con las 4 visualizaciones clave en una sola figura.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Dengue en Argentina - Dashboard Resumen',
             fontsize=20, fontweight='bold', y=1.01)

# --- Panel 1: Evolucion anual ---
ax1 = axes[0, 0]
ca = df.groupby(COL_ANIO)[COL_CANTIDAD].sum()
colores_bar = [COLOR_DENGUE if v == ca.max() else '#457B9D' for v in ca.values]
ax1.bar(ca.index, ca.values, color=colores_bar, edgecolor='white')
ax1.set_title('Evolucion anual', fontweight='bold')
ax1.set_ylabel('Casos')
ax1.tick_params(axis='x', rotation=45)

# --- Panel 2: Top provincias ---
ax2 = axes[0, 1]
df_clean = df[df['provincia_nombre'] != 'DESCONOCIDA']
cp = df_clean.groupby('provincia_nombre')[COL_CANTIDAD].sum().nlargest(8).sort_values()
ax2.barh(cp.index, cp.values, color=sns.color_palette('YlOrRd', len(cp)))
ax2.set_title('Top 8 provincias', fontweight='bold')
ax2.set_xlabel('Casos')

# --- Panel 3: Estacionalidad ---
ax3 = axes[1, 0]
cs = df.groupby(COL_SEMANA)[COL_CANTIDAD].sum()
ax3.fill_between(cs.index, cs.values, alpha=0.3, color=COLOR_DENGUE)
ax3.plot(cs.index, cs.values, color=COLOR_DENGUE, linewidth=2)
ax3.set_title('Estacionalidad (semana epidemiologica)', fontweight='bold')
ax3.set_xlabel('Semana')
ax3.set_ylabel('Casos')

# --- Panel 4: Grupos de edad ---
ax4 = axes[1, 1]
ce = df.groupby('grupo_edad_desc')[COL_CANTIDAD].sum().sort_values()
ax4.barh(ce.index, ce.values, color=sns.color_palette('viridis', len(ce)))
ax4.set_title('Distribucion por grupo de edad', fontweight='bold')
ax4.set_xlabel('Casos')
ax4.tick_params(axis='y', labelsize=8)

# Limpiar
for ax in axes.flat:
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
guardar_figura(fig, 'dashboard_resumen')
plt.show()

---
## Conclusiones

### Que nos dicen los datos?

1. **(completar)** - Sobre la tendencia temporal
2. **(completar)** - Sobre la distribucion geografica
3. **(completar)** - Sobre la estacionalidad
4. **(completar)** - Sobre los grupos de edad mas afectados

### Limitaciones del analisis

- Los datos dependen del registro y notificacion al SNVS, por lo que puede haber subregistro.
- El dataset usado cubre hasta (completar periodo).
- No se incluyen variables climaticas que podrian explicar los brotes.

### Posibles extensiones

- Incorporar datos climaticos (temperatura, precipitaciones) para analisis de correlacion.
- Construir un modelo predictivo de brotes por provincia.
- Crear un dashboard interactivo con Streamlit.

---

*Proyecto realizado por Fabiana Grisel Gonzalez - Data Quality Engineer.*
*Datos abiertos del Ministerio de Salud de la Nación Argentina (CC BY 4.0).*